# Contest workflow and reproducibility

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~30 min

## Read the task like a lawyer

- **Metric first**: F1 vs accuracy vs log-loss change which model wins. Optimize the leaderboard metric, not a convenient one.
- **Submission format**: column names, row order, id field, probability vs hard class. One wrong header and a perfect model scores zero.
- **Limits**: wall-clock time, RAM, no internet, allowed libraries, whether the jury re-runs your notebook.
- **Forbidden**: external data, pretrained weights, hand-labelling test rows, multiple accounts.

> **Note.** Print the sample submission and make yours `diff`-clean against it: same columns, same order, same index. Check `submission.shape` and `submission.columns` before uploading.



## Baseline first, then one change at a time

- Make a trivial valid submission first (constant prediction or the given baseline). It exercises the whole pipeline — if the plumbing is broken, you find out in 10 minutes, not on the last day.
- Build a **validation** set — a held-out slice of train that stands in for the hidden test — and mirror its split logic (by user, by time). Trust it more than the public leaderboard.
- Change one thing at a time and log the score of every experiment — the next step makes that a habit.



## Keep an experiment log

After run 10 you will not remember what run 3 did. A log is one row per experiment: what changed, validation score, seed, runtime. A list of dicts is enough — the discipline matters more than the tool.



In [ ]:
import pandas as pd
log = []
def record(name, val_score, **cfg):       # **cfg collects extra keyword arguments into a dict
    log.append({"name": name, "val": val_score, **cfg})   # **cfg spreads them into the new dict

record("baseline",   0.812, model="logreg")
record("more trees", 0.851, model="rf", n=500)
pd.DataFrame(log).sort_values("val", ascending=False)

The log turns "I think deeper trees helped" into a lookup — and it stops you re-tuning a knob you already swept.



## Reproducibility



In [ ]:
import random, numpy as np, torch
def seed_all(s=42):
    # Python, NumPy and PyTorch each keep a separate random generator —
    # seeding one does not seed the others, so all three calls are needed
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

Juries re-run notebooks. Fix seeds, note your `sklearn`/`torch` versions, and replace hard-coded local paths with relative ones (`Path("data") / "train.csv"`, not `C:\Users\you\...`). Time one full top-to-bottom run against the limit.

GPUs are not bit-deterministic — expect small numerical drift across hardware and leave margin in your score. If two runs with the same seed disagree wildly, you have hidden randomness: an unseeded shuffle, `set` iteration order, or `drop_last`-style nondeterminism in a loader.



### ✏️ Your validation score improves but the leaderboard score drops. Most likely cause?

**A.** Validation doesn't match the test distribution, or it leaks
**B.** The leaderboard metric weights errors differently than the local one
**C.** The submission file was uploaded with rows in the wrong order
**D.** The GPU gives slightly different numerical results on the test set

<details><summary>Answer</summary>

**A.** Validation doesn't match the test distribution, or it leaks

Validation must be built the way the test was built (grouped by user, split by time, etc.).

</details>

### ✏️ A notebook gives a great score but fails when re-run fresh top-to-bottom. Cause?

**A.** The metric used a different random seed by default
**B.** Restarting cleared the model's weight cache
**C.** Stale state from cells run out of order earlier
**D.** The GPU produced luckier randomness the first time

<details><summary>Answer</summary>

**C.** Stale state from cells run out of order earlier

Notebooks keep variables from deleted or edited cells; only a fresh-kernel run proves the result is real.

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

1. [Build a submission dict](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m00-readsubmission)
2. [Shuffled train/validation split](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-train-val-split)

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Build a submission dict

●○○ · ~8 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m00-readsubmission)

> Most olympiad submissions are 'id → prediction' mappings saved as CSV/JSON. Zipping two arrays into a dict sounds trivial — doing it fast and correctly (right order, right types, no off-by-one) under contest pressure is a real skill.

Implement `to_submission(ids, preds)` → dict mapping each id to its prediction, in the given order (values are compared numerically).

<details><summary>Hint</summary>

dict(zip(ids, preds)) — then convert numpy floats with float(v).

</details>


In [ ]:
import numpy as np

def to_submission(ids, preds):
    # your code
    return {}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(3, 20))\n    # random non-sequential ids — fabricating \"id_0..id_n\" or ignoring ids must fail\n    ids = [\"smp_%06d\" % v for v in rng.permutation(rng.integers(0, 10**6, n))]\n    return {\"ids\": ids, \"preds\": rng.normal(0, 1, n)}\n\ndef to_submission(ids, preds):\n    return {i: float(p) for i, p in zip(ids, preds)}", "to_submission", 20)

### Task 2 — Shuffled train/validation split

●○○ · ~8 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m01-train-val-split)

> Hold out a validation set before modelling: fit on train, check on validation, and keep the test set for the final estimate only. The standard recipe: permute the row indices with a seeded RNG, take the first `n_val` as validation, the rest as train. Seeding (`np.random.default_rng(seed)`) makes the split reproducible — with an unseeded split the score changes from run to run.

Implement `train_val_split(n, val_frac, seed)` → `(train_idx, val_idx)`.

Shuffle `0..n-1` with `np.random.default_rng(seed).permutation(n)`, let `n_val = int(round(n * val_frac))`, and return the last `n - n_val` indices as train, the first `n_val` as validation.

```
n=5, val_frac=0.4 → n_val=2, e.g. perm=[3,0,4,1,2] → val=[3,0], train=[4,1,2]
```

<details><summary>Hint</summary>

perm = rng.permutation(n); return perm[n_val:], perm[:n_val].

</details>


In [ ]:
import numpy as np

def train_val_split(n, val_frac, seed):
    rng = np.random.default_rng(seed)
    # your code
    return np.array([]), np.array([])

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(10, 60))\n    return {\"n\": n,\n            \"val_frac\": float(rng.choice([0.2, 0.25, 0.3, 0.4])),\n            \"seed\": int(rng.integers(0, 10000))}\n\ndef train_val_split(n, val_frac, seed):\n    rng = np.random.default_rng(seed)\n    perm = rng.permutation(n)\n    n_val = int(round(n * val_frac))\n    return perm[n_val:], perm[:n_val]", "train_val_split", 20)

---

**Go deeper**
- [IOAI official task repos](https://github.com/IOAI-official)